In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="management_pnl",
    choices=["management_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

checkpoint = 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/checkpoints/'

In [0]:
%sql
select * from fq_dev_pnl_catalog.bronze.gl_report limit 1

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Formula & Functions Management P&L"

In [0]:
def enrich_data(df):
    # df = spark.read.table('default.postman_response_january')
    df_exploded = df.select(explode('results').alias('result')).select('result.*')

    # ✅ Reverse sign when accountType == 'Income'
    df_exploded = df_exploded.withColumn(
        "amount",
        when(col("accountType") == "Income", -col("amount"))
        .when(col("accountName") == "75704 Talabat- Commission Discount", -col("amount"))
        .when(col("accountName") == "75713 Noon- Commission Discount", -col("amount"))
        .otherwise(col("amount"))
    ).withColumn(
        "year", col('year').cast('int')
    )

    df_brand_ho_rows = spark.read.table('fq_dev_pnl_catalog.bronze.brand_ho_allocation_cost')

    df_exploded_keys = df_exploded.select(
        col("year"), 
        col("month"), 
        col("location").alias("netsuite_location_name")
    ).distinct()


    df_brand_ho_rows_filtered = df_brand_ho_rows.join(
        df_exploded_keys,
        (df_brand_ho_rows["year"] == df_exploded_keys["year"]) &
        (df_brand_ho_rows["month"] == df_exploded_keys["month"]) &
        (df_brand_ho_rows["location"] == df_exploded_keys["netsuite_location_name"]),
        "inner"
    ).select(df_brand_ho_rows["*"])

    # Perform union with filtered data
    df_exploded = df_exploded.union(df_brand_ho_rows_filtered)

    df_coa_master = spark.read.table("bronze.dim_coa_master")
    # df_coa_master = to_snake_case_df(df_coa_master)
    df_location_master = spark.read.table("bronze.dim_location_master")
    # df_location_master = to_snake_case_df(df_location_master)
    # df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping


    # for column in df_coa_master.columns:
    #     if dict(df_coa_master.dtypes)[column] == 'string':
    #         df_coa_master = df_coa_master.withColumn(column, trim(col(column)))

    df_all_masters = df_exploded.join(
            df_coa_master, 
            df_coa_master["account_number"].cast("string") == df_exploded["accountNo"], 
            'inner'
        ).join(
            df_location_master,
            col("location") == df_location_master.netsuite_location_name,
            'left'
        )

    # Exclude summary "Total" accounts that duplicate management group totals
    df_all_masters_filtered = df_all_masters.filter(
        ~col("account_name").startswith("Total ")
    )

    return final_df(df_all_masters_filtered)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_pnl_catalog.silver.management_pnl (
  city STRING,
  management_sort_order INT,
  location_id INT,
  store_type STRING,
  major_group STRING,
  detail_total STRING COMMENT 'Detail/Total indicator',
  account_name STRING,
  zone STRING,
  sub_group STRING,
  management_details_total STRING,
  type STRING COMMENT 'Store/HO type',
  account_type STRING COMMENT 'Income/Expense type',
  store_open_date2 STRING,
  brand_id STRING,
  company_id STRING,
  parent_company STRING,
  country_code STRING,
  group_name STRING COMMENT 'Sales and Services Income group',
  management_group STRING,
  year INT,
  month STRING,
  netsuite_location_name STRING,
  mapped_name STRING,
  amount DECIMAL(18,2),
  budget_amount DECIMAL(18,2),
  py_amount DECIMAL(18,2) COMMENT 'Previous year amount'
)
USING DELTA
CLUSTER BY auto
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/silver/management_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
%python
def merge_stream_management_pnl(df, i):
    try:
        # Enrich data to include all necessary fields
        management_pnl_upsert = enrich_data(df)
        management_pnl_upsert.createOrReplaceTempView("management_pnl_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.management_pnl target
            USING (
                SELECT 
                    *
                FROM management_pnl_upsert_microbatch
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.netsuite_location_name = source.netsuite_location_name
                AND target.mapped_name = source.mapped_name
            WHEN MATCHED THEN UPDATE SET
                target.amount = source.amount
            WHEN NOT MATCHED THEN INSERT (
                city,
                management_sort_order,
                location_id,
                store_type,
                major_group,
                `Detail/Total`,
                account_name,
                zone,
                sub_group,
                management_details_total,
                type,
                account_type,
                store_open_date2,
                brand_id,
                company_id,
                parent_company,
                country_code,
                `group`,
                management_group,
                year,
                month,
                netsuite_location_name,
                mapped_name,
                amount
            ) VALUES (
                source.city,
                source.management_sort_order,
                source.location_id,
                source.store_type,
                source.major_group,
                source.`Detail/Total`,
                source.account_name,
                source.zone,
                source.sub_group,
                source.management_details_total,
                source.type,
                source.account_type,
                source.store_open_date2,
                source.brand_id,
                source.company_id,
                source.parent_company,
                source.country_code,
                source.`group`,
                source.management_group,
                source.year,
                source.month,
                source.netsuite_location_name,
                source.mapped_name,
                source.amount
            )
        """)
        
        print(f"Successfully merged batch {i}")
        print(f"Batch {i}: {df.count()} rows")
        display(df.limit(10))  # Show first 10 rows

    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

# Streaming read and write
(spark.readStream
    .table("fq_dev_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_management_pnl)
    .option("mergeSchema", "true")
    #.option('skipChangeCommits', "true") #the stream ignores commits that modify existing data and only processes newly appended files.
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_management_pnl3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  count(distinct month) as total_months,
  count(distinct year) as total_years,
  count(distinct netsuite_location_name) as total_locations
FROM fq_dev_catalog.silver.management_pnl;

In [0]:
%sql
select * from fq_dev_catalog.silver.management_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()